# StressRisk AI: Final 19-Feature Training Workflow

This notebook trains and evaluates Random Forest, Logistic Regression, SVM, and XGBoost with one reproducible preprocessing pipeline. Meal Regularity and Meditation Practice are intentionally excluded.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError("XGBoost is required to run this workflow.") from exc

RANDOM_STATE = 42
BASE_DIR = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_PATH = BASE_DIR / "Dataset" / "StressRisk_AI_lifestyle and risk prediction dataset.csv"
MODEL_DIR = BASE_DIR / "saved_model"
ANALYSIS_DIR = BASE_DIR / "analysis"
TARGET = "Stress Level"
REMOVED_FEATURES = {"Meal Regularity", "Meditation Practice"}
IDENTIFIER_COLUMNS = {"Person ID", "erson ID", "index"}

data = pd.read_csv(DATA_PATH)
if TARGET not in data.columns:
    raise ValueError(f"Missing target column: {TARGET}")
columns_to_drop = (IDENTIFIER_COLUMNS | REMOVED_FEATURES) & set(data.columns)
model_data = data.drop(columns=sorted(columns_to_drop)).dropna(subset=[TARGET]).copy()
feature_columns = [column for column in model_data.columns if column != TARGET]
if TARGET in feature_columns or REMOVED_FEATURES.intersection(feature_columns):
    raise ValueError("Target or excluded features leaked into the model inputs.")
if len(feature_columns) != 19:
    raise ValueError(f"Expected exactly 19 input features, found {len(feature_columns)}: {feature_columns}")

X = model_data[feature_columns]
y = model_data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [column for column in feature_columns if column not in numeric_features]
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "SVM": SVC(
        kernel="rbf", probability=True, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=250, max_depth=5, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9, objective="multi:softprob",
        eval_metric="mlogloss", random_state=RANDOM_STATE, n_jobs=2,
    ),
}

results = []
fitted_models = {}
for name, estimator in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    fit_target = y_train
    decode_prediction = lambda values: values
    if name == "XGBoost":
        xgb_classes = sorted(y_train.unique())
        xgb_encoding = {label: index for index, label in enumerate(xgb_classes)}
        fit_target = y_train.map(xgb_encoding)
        decode_prediction = lambda values: [xgb_classes[int(value)] for value in values]
    pipeline.fit(X_train, fit_target)
    predictions = decode_prediction(pipeline.predict(X_test))
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, predictions, average="weighted", zero_division=0),
        "F1-score": f1_score(y_test, predictions, average="weighted", zero_division=0),
    })
    fitted_models[name] = pipeline

model_comparison = pd.DataFrame(results)
ANALYSIS_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
model_comparison.to_csv(ANALYSIS_DIR / "model_comparison_results.csv", index=False)
random_forest_model = fitted_models["Random Forest"]
joblib.dump(random_forest_model, MODEL_DIR / "stressrisk_model.pkl")
joblib.dump(feature_columns, MODEL_DIR / "feature_columns.pkl")
reloaded_model = joblib.load(MODEL_DIR / "stressrisk_model.pkl")
reloaded_features = joblib.load(MODEL_DIR / "feature_columns.pkl")
assert reloaded_features == feature_columns
assert len(reloaded_features) == 19
sample_prediction = reloaded_model.predict(X_test.iloc[[0]])[0]
print("Final input features:", len(feature_columns))
print("Features:", feature_columns)
print(model_comparison.to_string(index=False))
print("Reloaded Random Forest prediction:", sample_prediction)
print("Saved model:", MODEL_DIR / "stressrisk_model.pkl")
print("Saved feature columns:", MODEL_DIR / "feature_columns.pkl")

Final input features: 19
Features: ['Gender', 'Age', 'Occupation', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 'Heart Rate', 'Daily Steps', 'Systolic BP', 'Diastolic BP', 'BMI', 'BMI Category', 'Working Hours', 'Screen Time', 'Water Intake', 'Caffeine Intake', 'Social Interaction', 'Smoking Habit', 'Alcohol Intake']
              Model  Accuracy  Precision  Recall  F1-score
      Random Forest     1.000   1.000000   1.000   1.00000
Logistic Regression     0.980   0.979935   0.980   0.97991
                SVM     0.998   0.998041   0.998   0.99800
            XGBoost     1.000   1.000000   1.000   1.00000
Reloaded Random Forest prediction: 10
Saved model: c:\Users\mohit\OneDrive\Documents\Desktop\Stress_Risk_Project sem 3 demo main project\saved_model\stressrisk_model.pkl
Saved feature columns: c:\Users\mohit\OneDrive\Documents\Desktop\Stress_Risk_Project sem 3 demo main project\saved_model\feature_columns.pkl
